# Adım 6: ML Training
Modelleri eğit ve MLflow'a gönder.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor, GBTRegressor, GeneralizedLinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import time

spark = SparkSession.builder \
    .appName("ML") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("NYC_Taxi_Fare_Prediction")

df = spark.read.format("delta").load("/app/delta/taxi_silver")

feature_cols = ["pickup_hour", "day_of_week", "is_weekend", "trip_distance", "trip_duration", "is_airport"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vect = assembler.transform(df)

train_data, test_data = df_vect.randomSplit([0.8, 0.2], seed=42)

models = [
    (LinearRegression(labelCol="fare_amount"), "Linear_Regression"),
    (DecisionTreeRegressor(labelCol="fare_amount"), "Decision_Tree"),
    (RandomForestRegressor(labelCol="fare_amount"), "Random_Forest"),
    (GBTRegressor(labelCol="fare_amount"), "GBT_Regressor"),
    (GeneralizedLinearRegression(labelCol="fare_amount"), "GLR_Regressor")
]

best_r2 = -1
best_predictions = None
best_model_name = ""

for model_obj, name in models:
    with mlflow.start_run(run_name=name):
        print(f"\nEğitiliyor: {name}...")
        
        model = model_obj.setFeaturesCol("features").fit(train_data)
        predictions = model.transform(test_data)
        
        predictions = predictions.withColumn("residual", col("fare_amount") - col("prediction"))
        
        evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
        evaluator_mae = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="mae")
        evaluator_r2 = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="r2")
        
        rmse = evaluator_rmse.evaluate(predictions)
        mae = evaluator_mae.evaluate(predictions)
        r2 = evaluator_r2.evaluate(predictions)
        
        mlflow.log_param("model_type", name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        
        if hasattr(model, 'featureImportances'):
            importances = model.featureImportances.toArray()
            for i, feat in enumerate(feature_cols):
                mlflow.log_metric(f"importance_{feat}", float(importances[i]))
        elif hasattr(model, 'coefficients'):
            coeffs = model.coefficients.toArray()
            for i, feat in enumerate(feature_cols):
                mlflow.log_metric(f"importance_{feat}", abs(float(coeffs[i])))
        
        mlflow.spark.log_model(model, f"{name}_model")
        
        print(f"{name} Sonuçları -> R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

        if r2 > best_r2:
            best_r2 = r2
            best_predictions = predictions
            best_model_name = name

print(f"\n🏆 EN BAŞARILI MODEL SEÇİLDİ: {best_model_name} (R2: {best_r2:.4f})")
print("GOLD KATMANINA YAZILIYOR...")

best_predictions.write.format("delta").mode("overwrite").save("/app/delta/taxi_gold")

print(f"Başarı! {best_model_name} sonuçları /app/delta/taxi_gold dizinine kaydedildi.")
print("\nTÜM MODELLER EĞİTİLDİ VE MLFLOW'A KAYDEDİLDİ.")


Eğitiliyor: Linear_Regression...
Linear_Regression Sonuçları - R2: 0.5413
🏃 View run Linear_Regression at: http://mlflow:5000/#/experiments/1/runs/b8fb4a4a1c0c409ba2bfa84b4a3541aa
🧪 View experiment at: http://mlflow:5000/#/experiments/1

Eğitiliyor: Decision_Tree...
Decision_Tree Sonuçları - R2: 0.8634
🏃 View run Decision_Tree at: http://mlflow:5000/#/experiments/1/runs/e13110494dce4ea89faacf17660efdb5
🧪 View experiment at: http://mlflow:5000/#/experiments/1

Eğitiliyor: Random_Forest...
Random_Forest Sonuçları - R2: 0.8250
🏃 View run Random_Forest at: http://mlflow:5000/#/experiments/1/runs/f6f7d5e37145446692da0159b3e334fb
🧪 View experiment at: http://mlflow:5000/#/experiments/1

Eğitiliyor: GBT_Regressor...
GBT_Regressor Sonuçları - R2: 0.8434
🏃 View run GBT_Regressor at: http://mlflow:5000/#/experiments/1/runs/2a770a90a902429b93cbc2ff74570eda
🧪 View experiment at: http://mlflow:5000/#/experiments/1

Eğitiliyor: GLR_Regressor...
GLR_Regressor Sonuçları - R2: 0.5413
🏃 View run GLR_Re